Implementing the AdaBoost Algorithm





In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [8]:
class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.models = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        w = np.ones(n_samples) / n_samples

        for _ in range(self.n_estimators):
            model = DecisionTreeClassifier(max_depth=1)
            model.fit(X, y, sample_weight=w)
            predictions = model.predict(X)

            # Convert y from {0, 1} to {-1, 1} for AdaBoost's error calculation
            y_transformed = np.where(y == 0, -1, 1)
            predictions_transformed = np.where(predictions == 0, -1, 1)

            err = np.sum(w * (predictions_transformed != y_transformed)) / np.sum(w)

            # Handle cases where err is 0 or 1 to prevent log(0) or division by zero
            # A very small epsilon is added to avoid division by zero and log of zero/negative numbers.
            err = np.clip(err, 1e-10, 1 - 1e-10)
            alpha = 0.5 * np.log((1 - err) / err)

            self.models.append(model)
            self.alphas.append(alpha)

            # Update weights, using transformed y and predictions
            w *= np.exp(-alpha * y_transformed * predictions_transformed)
            w /= np.sum(w)

    def predict(self, X):
        strong_preds = np.zeros(X.shape[0])

        for model, alpha in zip(self.models, self.alphas):
            predictions = model.predict(X)
            # Transform predictions to -1, 1 for consistent voting
            predictions_transformed = np.where(predictions == 0, -1, 1)
            strong_preds += alpha * predictions_transformed

        # Final prediction based on the sign of the weighted sum, transformed back to 0, 1
        return np.where(np.sign(strong_preds) == -1, 0, 1)

In [7]:
if __name__ == "__main__":

    # Generate synthetic data with two classes
    X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Initialize and train the AdaBoost classifier
    adaboost = AdaBoost(n_estimators=50)
    adaboost.fit(X_train, y_train)

    # Make predictions on the test set
    predictions = adaboost.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    # Calculate ROC-AUC score. Note: roc_auc_score requires probability scores or decision function scores.
    # Since our predict method returns hard classes (0 or 1), we use a try-except block.
    # For a proper ROC-AUC, the predict method would need to return probabilities or a decision_function output.
    try:
        # Ensure y_test and predictions are in the correct format for roc_auc_score
        roc_auc = roc_auc_score(y_test, predictions)
    except ValueError:
        roc_auc = 'Undefined (requires probability scores for meaningful calculation)'

    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(f"ROC-AUC: {roc_auc}")

Accuracy: 84.67%
Precision: 0.86
Recall: 0.84
F1 Score: 0.85
ROC-AUC: 0.8469410456062292
